## 🎯 Learning Objectives
* Successfully implement a multi-layer perceptron (MLP) using PyTorch's nn.Module for image classification.
* Efficiently load, transform, and batch the MNIST dataset using torchvision and DataLoader.
* Configure and apply appropriate loss functions (CrossEntropyLoss) and optimizers (Adam) for deep learning tasks.
* Construct a complete training loop in PyTorch, encompassing forward pass, backward pass, and parameter updates.
* Evaluate the performance of a trained neural network on unseen data, reporting key metrics like accuracy.


## DL01-L08: Exercise: Build and Train a Feedforward Network on MNIST

### Context
In previous lessons, we've explored the foundational concepts of feedforward neural networks, understood how they learn through backpropagation, and familiarized ourselves with PyTorch's core components like `torch.Tensor`, `nn.Module`, `nn.Linear`, and activation functions. We've seen how these building blocks come together to form powerful models capable of learning complex patterns. Now, it's time to put that knowledge into practice by implementing a complete deep learning workflow.

### Your Task
Your mission is to build and train a simple feedforward neural network (also known as a Multi-Layer Perceptron or MLP) to classify handwritten digits from the famous MNIST dataset. This exercise will solidify your understanding of the entire deep learning workflow in PyTorch, from data loading and preprocessing to model definition, training, and evaluation.

### Requirements
To successfully complete this exercise, your solution must adhere to the following specifications:

1.  **Network Architecture:**
    *   Define your neural network as a class that inherits from `torch.nn.Module`.
    *   It must include at least one hidden layer.
    *   Use `torch.nn.Linear` for all fully connected layers.
    *   Apply a non-linear activation function (e.g., `torch.nn.ReLU`) between hidden layers to introduce non-linearity.
    *   The output layer should produce raw scores (logits) for 10 classes (representing digits 0-9).

2.  **Data Handling:**
    *   Load the MNIST dataset using `torchvision.datasets.MNIST` for both training and testing splits.
    *   Apply necessary transformations to the images, including converting them to tensors (`transforms.ToTensor()`) and normalizing their pixel values (`transforms.Normalize()`). Standard MNIST normalization values are `(0.1307,)` for mean and `(0.3081,)` for standard deviation.
    *   Utilize `torch.utils.data.DataLoader` for efficient batching and shuffling of the training data.

3.  **Training Loop:**
    *   Define the loss function: `torch.nn.CrossEntropyLoss` is appropriate for multi-class classification with logits.
    *   Choose an optimizer: `torch.optim.Adam` is highly recommended for its efficiency and robustness.
    *   Implement a training loop that iterates over a specified number of epochs and processes data in batches.
    *   Within each training step, perform the forward pass, calculate the loss, execute the backward pass (backpropagation), and update the model's weights using the optimizer.
    *   Ensure data and the model are moved to the appropriate device (CPU or GPU if available) for optimal performance.

4.  **Evaluation:**
    *   Implement a separate evaluation loop to assess your model's performance on the unseen test dataset.
    *   Calculate and report the final accuracy of your model on the test set.
    *   Remember to set the model to evaluation mode (`model.eval()`) and disable gradient calculations (`torch.no_grad()`) during evaluation.

### Evaluation Criteria
Your solution will be evaluated based on the following:

*   **Correctness:** Does the code run without errors and produce reasonable classification results?
*   **Completeness:** Are all the requirements listed above fully met?
*   **Clarity and Readability:** Is the code well-structured, easy to understand, and appropriately commented where necessary?
*   **PyTorch Best Practices:** Does the code follow idiomatic PyTorch patterns and conventions (e.g., proper device handling, `model.train()`/`model.eval()`, `optimizer.zero_grad()`, `torch.no_grad()`)?


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

# --- Configuration and Hyperparameters ---
# Determine if a GPU is available and set the device accordingly
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Hyperparameters for the model and training process
input_size = 28 * 28  # MNIST images are 28x28 pixels
hidden_size = 128     # Number of neurons in the hidden layer
num_classes = 10      # MNIST has 10 classes (digits 0-9)
num_epochs = 10       # Number of full passes over the training dataset
batch_size = 64       # Number of samples per gradient update
learning_rate = 0.001 # Step size for the optimizer

# --- Data Loading and Preprocessing ---
# Define transformations to apply to the MNIST images
# 1. ToTensor: Converts a PIL Image or numpy.ndarray to a PyTorch FloatTensor.
#    Scales pixel values from [0, 255] to [0.0, 1.0].
# 2. Normalize: Normalizes a tensor image with mean and standard deviation.
#    MNIST standard values are (0.1307,) for mean and (0.3081,) for std dev.
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

# Load the MNIST training dataset
# download=True will download the dataset if it's not already present
train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)

# Load the MNIST test dataset
test_dataset = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

# Create DataLoaders for efficient batching and shuffling
# train_loader shuffles the data at each epoch for better generalization
train_loader = DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle=True)

# test_loader does not need shuffling as order doesn't affect evaluation
test_loader = DataLoader(dataset=test_dataset, batch_size=batch_size, shuffle=False)

# --- Verify Data Loading (Optional) ---
print(f"\nTraining dataset size: {len(train_dataset)} samples")
print(f"Test dataset size: {len(test_dataset)} samples")

# Get one batch to inspect its shape
for images, labels in train_loader:
    print(f"Shape of one batch of images: {images.shape}") # Expected: [batch_size, 1, 28, 28]
    print(f"Shape of one batch of labels: {labels.shape}") # Expected: [batch_size]
    break

# Plot a few images from the training set (optional)
fig, axes = plt.subplots(1, 5, figsize=(10, 2))
for i in range(5):
    # Denormalize for display: image = image * std + mean
    img = images[i].squeeze().cpu() * 0.3081 + 0.1307
    axes[i].imshow(img, cmap='gray')
    axes[i].set_title(f"Label: {labels[i].item()}")
    axes[i].axis('off')
plt.suptitle("Sample MNIST Images")
plt.show()


### Your Implementation

It's your turn to shine! Below, implement your solution for building and training a feedforward neural network on the MNIST dataset. Refer to the requirements above and the setup code provided. You should define your `FeedForwardNet` class, instantiate the model, define the loss and optimizer, and then implement the full training and evaluation loops.

Good luck, and remember to break down the problem into smaller, manageable parts!


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

# --- Configuration and Hyperparameters (re-defined for self-contained solution) ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
input_size = 28 * 28
hidden_size = 128
num_classes = 10
num_epochs = 10
batch_size = 64
learning_rate = 0.001

# --- Data Loading and Preprocessing (re-defined for self-contained solution) ---
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])
train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root='./data', train=False, download=True, transform=transform)
train_loader = DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(dataset=test_dataset, batch_size=batch_size, shuffle=False)

# --- 1. Define the FeedForward Neural Network (MLP) ---
class FeedForwardNet(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes):
        # Call the constructor of the parent class (nn.Module)
        super(FeedForwardNet, self).__init__()
        
        # First fully connected layer: input_size -> hidden_size
        self.fc1 = nn.Linear(input_size, hidden_size)
        
        # ReLU activation function for non-linearity
        self.relu = nn.ReLU()
        
        # Second fully connected layer: hidden_size -> num_classes
        # This layer outputs the raw scores (logits) for each class
        self.fc2 = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        # Flatten the input image from 2D (28x28) to 1D (784)
        # x.shape will be [batch_size, 1, 28, 28], we want [batch_size, 784]
        # The -1 in reshape infers the batch size automatically
        x = x.reshape(-1, input_size)
        
        # Pass through the first linear layer
        out = self.fc1(x)
        
        # Apply ReLU activation
        out = self.relu(out)
        
        # Pass through the second linear layer to get logits
        out = self.fc2(out)
        return out

# --- 2. Instantiate the Model, Loss Function, and Optimizer ---
# Create an instance of our FeedForwardNet and move it to the specified device (CPU/GPU)
model = FeedForwardNet(input_size, hidden_size, num_classes).to(device)

# Define the loss function: CrossEntropyLoss is suitable for multi-class classification
# It combines nn.LogSoftmax() and nn.NLLLoss() in one single class.
# It expects raw logits as input and integer class labels as target.
criterion = nn.CrossEntropyLoss()

# Define the optimizer: Adam is a popular choice for its adaptive learning rate capabilities
# It will optimize the parameters of our model with the specified learning rate.
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

# --- 3. Training Loop ---
print("\nStarting training...")
for epoch in range(num_epochs):
    # Set the model to training mode
    # This enables features like Dropout (if used) and Batch Normalization updates.
    model.train()
    
    for i, (images, labels) in enumerate(train_loader):
        # Move images and labels to the configured device
        # Images are flattened here to match the input_size of the network
        images = images.reshape(-1, input_size).to(device)
        labels = labels.to(device)
        
        # Forward pass: Compute predicted outputs by passing inputs to the model
        outputs = model(images)
        
        # Calculate the loss: Compare predicted outputs with true labels
        loss = criterion(outputs, labels)
        
        # Backward pass: Compute gradient of the loss with respect to model parameters
        # This calculates dL/dw for all weights w
        optimizer.zero_grad() # Clear previous gradients before computing new ones
        loss.backward()       # Perform backpropagation
        
        # Optimizer step: Update model parameters using the computed gradients
        optimizer.step()
        
        # Print training progress
        if (i+1) % 100 == 0:
            print (f'Epoch [{epoch+1}/{num_epochs}], Step [{i+1}/{len(train_loader)}], Loss: {loss.item():.4f}')

print("Training complete.")

# --- 4. Evaluation Loop ---
# Set the model to evaluation mode
# This disables features like Dropout and ensures Batch Normalization uses running means/variances.
model.eval()

# Disable gradient calculations during evaluation
# This saves memory and speeds up computations as we don't need gradients for inference.
with torch.no_grad():
    correct = 0
    total = 0
    for images, labels in test_loader:
        # Move images and labels to the configured device
        images = images.reshape(-1, input_size).to(device)
        labels = labels.to(device)
        
        # Forward pass to get predictions
        outputs = model(images)
        
        # Get the predicted class with the highest score
        # torch.max returns (values, indices). We need the indices.
        _, predicted = torch.max(outputs.data, 1)
        
        # Update total number of images and correct predictions
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    # Calculate and print the final accuracy
    accuracy = 100 * correct / total
    print(f'\nAccuracy of the network on the {total} test images: {accuracy:.2f}%')

# --- Optional: Save the trained model ---
# torch.save(model.state_dict(), 'mnist_ffn_model.pth')
# print("Model saved to mnist_ffn_model.pth")
